## EXP1 wedge comparison (CutSky vs Stage-3)

This notebook compares the **three EXP1 wedges** used in the controlled experiments:

- **CutSky BGS wedge**: `exp1_cutsky_bgs_z025_030_parentbgs20260527`
- **CutSky intersection wedge** (same halos as stage-3): `exp1_cutsky_intersection_z025_030_parentbgs20260527`
- **Stage-3 unique wedge**: `exp1_stage3_unique_ra120_140_dec16p5_26p7_z025_030_rs7`

It loads:
- positions from `graph_constructions/<prefix>_metadata.json` (`points_xyz`)
- labels (CWEB) + eigenvalues from the corresponding `sbi_caches/<prefix>_sbi_cache.pkl`

and renders a side-by-side 3D Plotly view with consistent coloring.


In [ ]:
from __future__ import annotations

import os

# These caches contain JAX arrays inside the pickle.
# Force CPU backend so the notebook works on login/CPU nodes.
os.environ.setdefault("JAX_PLATFORMS", "cpu")
os.environ.setdefault("JAX_PLATFORM_NAME", "cpu")

import json
import pickle
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

GRAPH_DIR = Path("/pscratch/sd/d/dkololgi/abacus/graph_constructions").resolve()
CACHE_DIR = Path("/pscratch/sd/d/dkololgi/abacus/sbi_caches").resolve()

CUTSKY_BGS = "exp1_cutsky_bgs_z025_030_parentbgs20260527"
CUTSKY_INTERSECTION = "exp1_cutsky_intersection_z025_030_parentbgs20260527"

# Stage-3 EXP1 wedge prefix (graph topology / points_xyz etc.)
STAGE3_UNIQUE = "exp1_stage3_unique_ra120_140_dec16p5_26p7_z025_030_rs7"

# By default, use the re-annotated stage-3 cache (halo-xcom T-web assignment).
# To switch back to the original stage-3 cache, comment the first line and uncomment the second.
STAGE3_CACHE_SUFFIX = "_sbi_cache_halo_xcom.pkl"
# STAGE3_CACHE_SUFFIX = "_sbi_cache.pkl"

PREFIXES = [
    ("CutSky BGS", CUTSKY_BGS),
    ("CutSky ∩ Stage-3", CUTSKY_INTERSECTION),
    ("Stage-3 unique", STAGE3_UNIQUE),
]

LAMBDA_THR = 0.2  # CWEB threshold convention
N_PLOT = 50_000   # max points per panel (random subsample)
SEED = 0


In [ ]:
@dataclass
class WedgeData:
    label: str
    prefix: str
    xyz: np.ndarray  # (N,3)
    cweb: np.ndarray  # (N,)
    lam: np.ndarray  # (N,3)


def _load_xyz(prefix: str) -> np.ndarray:
    meta_path = GRAPH_DIR / f"{prefix}_metadata.json"
    with meta_path.open("r", encoding="utf-8") as f:
        meta = json.load(f)
    xyz_path = GRAPH_DIR / meta["files"]["points_xyz"]
    xyz = np.load(xyz_path)
    xyz = np.asarray(xyz, dtype=np.float64)
    if xyz.ndim != 2 or xyz.shape[1] != 3:
        raise ValueError(f"points_xyz must be (N,3); got {xyz.shape} at {xyz_path}")
    return xyz


def _cache_path_for_prefix(prefix: str) -> Path:
    if prefix == STAGE3_UNIQUE:
        return CACHE_DIR / f"{prefix}{STAGE3_CACHE_SUFFIX}"
    return CACHE_DIR / f"{prefix}_sbi_cache.pkl"


def _load_cache(prefix: str) -> dict:
    cache_path = _cache_path_for_prefix(prefix)
    with cache_path.open("rb") as f:
        return pickle.load(f)


def _cweb_from_lam(lam: np.ndarray, thr: float) -> np.ndarray:
    # same convention as used elsewhere: class = number of lambdas > thr (0..3)
    return (lam > thr).sum(axis=1).astype(np.int32)


def load_wedge(label: str, prefix: str, thr: float) -> WedgeData:
    xyz = _load_xyz(prefix)
    data = _load_cache(prefix)

    lam = np.asarray(data["eigenvalues_raw"], dtype=np.float64)
    if lam.shape[0] != xyz.shape[0] or lam.shape[1] != 3:
        raise ValueError(f"Mismatch: xyz={xyz.shape} eigenvalues_raw={lam.shape} for {prefix}")

    # Prefer cache-provided labels if present, else derive from eigenvalues.
    if "classification_labels" in data and data["classification_labels"] is not None:
        cweb = np.asarray(data["classification_labels"], dtype=np.int32)
    else:
        cweb = _cweb_from_lam(lam, thr)

    if cweb.shape[0] != xyz.shape[0]:
        raise ValueError(f"Mismatch: xyz={xyz.shape} cweb={cweb.shape} for {prefix}")

    return WedgeData(label=label, prefix=prefix, xyz=xyz, cweb=cweb, lam=lam)


wedges = [load_wedge(lbl, pfx, LAMBDA_THR) for (lbl, pfx) in PREFIXES]
[(w.label, w.prefix, w.xyz.shape[0]) for w in wedges]


In [ ]:
COLORS = {
    0: "#440154",  # void
    1: "#3b528b",  # wall
    2: "#21918c",  # filament
    3: "#fde725",  # cluster
}


def _subsample(xyz: np.ndarray, cweb: np.ndarray, lam: np.ndarray, n_plot: int, seed: int):
    n = xyz.shape[0]
    if n <= n_plot:
        idx = np.arange(n)
    else:
        rng = np.random.default_rng(seed)
        idx = rng.choice(n, size=n_plot, replace=False)
        idx.sort()
    return xyz[idx], cweb[idx], lam[idx]


def _trace_for_class(xyz: np.ndarray, cls: np.ndarray, klass: int, name: str):
    m = cls == klass
    if not np.any(m):
        return None
    pts = xyz[m]
    return go.Scatter3d(
        x=pts[:, 0],
        y=pts[:, 1],
        z=pts[:, 2],
        mode="markers",
        name=name,
        marker=dict(size=1.5, color=COLORS[klass], opacity=0.55),
        showlegend=True,
    )


fig = make_subplots(
    rows=1,
    cols=3,
    specs=[[{"type": "scene"}, {"type": "scene"}, {"type": "scene"}]],
    subplot_titles=[
        f"{wedges[0].label}<br><sup>N={wedges[0].xyz.shape[0]:,} | thr={LAMBDA_THR}</sup>",
        f"{wedges[1].label}<br><sup>N={wedges[1].xyz.shape[0]:,} | thr={LAMBDA_THR}</sup>",
        f"{wedges[2].label}<br><sup>N={wedges[2].xyz.shape[0]:,} | thr={LAMBDA_THR}</sup>",
    ],
)

for j, w in enumerate(wedges, start=1):
    xyz_s, cweb_s, lam_s = _subsample(w.xyz, w.cweb, w.lam, N_PLOT, seed=SEED + j)
    for klass, cname in [(0, "void"), (1, "wall"), (2, "filament"), (3, "cluster")]:
        tr = _trace_for_class(xyz_s, cweb_s, klass, f"{cname}")
        if tr is not None:
            fig.add_trace(tr, row=1, col=j)

# Consistent scene styling
scene_common = dict(
    xaxis_title="x (Mpc)",
    yaxis_title="y (Mpc)",
    zaxis_title="z (Mpc)",
    aspectmode="data",
)
fig.update_layout(
    height=650,
    width=1400,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)
fig.update_layout(scene=scene_common, scene2=scene_common, scene3=scene_common)

fig.show()
